# Part 5: Tunning 
## 5.1 Hyperparameter Tuning:  Jointly on Representation and Model

In [5]:
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from src.config import cv_strategy, combined_preprocessor, target, text_feature, structured_features
import pandas as pd

# Load Data 
df = pd.read_csv("../data/processed/train_clean.csv")

X = df[[text_feature] + structured_features].copy()
y = df[target].copy()

X[text_feature] = X[text_feature].fillna("")


# HistGradientBoosting is used as the boosted model.
# It supports early stopping and has model parameters such as learning_rate and max_depth.
tuning_pipe = Pipeline([
    ("preprocessor", combined_preprocessor),
    ("model", HistGradientBoostingClassifier(
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

pipeline_params = {
    # Text representation: TF-IDF
    "preprocessor__text__tfidf__max_features": [1000, 2000, 5000],
    "preprocessor__text__tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "preprocessor__text__tfidf__min_df": [2, 5, 10],

    # Text representation: SVD
    "preprocessor__text__svd__n_components": [50, 100, 200],

    # Model: boosted model hyperparameters
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_iter": [100, 200, 300],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__max_depth": [3, 5, 7],
    "model__l2_regularization": [0.0, 0.01, 0.1]
}

random_search = RandomizedSearchCV(
    estimator=tuning_pipe,
    param_distributions=pipeline_params,
    n_iter=20,
    scoring="f1_weighted",
    cv=cv_strategy,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
    error_score="raise"
)

print("Starting RandomizedSearchCV for joint representation and model tuning...")

random_search.fit(X, y)

print("\nBest Cross-Validated Weighted F1:")
print(round(random_search.best_score_, 4))

print("\nBest Parameters:")
for param, value in random_search.best_params_.items():
    print(param, ":", value)

best_tuned_model = random_search.best_estimator_

best_params_df = pd.DataFrame([
    {
        "Parameter": param,
        "Best Value": value
    }
    for param, value in random_search.best_params_.items()
])

display(best_params_df)

tuning_results = pd.DataFrame(random_search.cv_results_)

top_tuning_results = tuning_results[
    [
        "mean_test_score",
        "std_test_score",
        "mean_train_score",
        "std_train_score",
        "param_preprocessor__text__tfidf__max_features",
        "param_preprocessor__text__tfidf__ngram_range",
        "param_preprocessor__text__tfidf__min_df",
        "param_preprocessor__text__svd__n_components",
        "param_model__learning_rate",
        "param_model__max_iter",
        "param_model__max_leaf_nodes",
        "param_model__max_depth",
        "param_model__l2_regularization"
    ]
].sort_values(
    by="mean_test_score",
    ascending=False
)

print("\nTop 10 Tuning Combinations:")
display(top_tuning_results.head(10).round(4))

representation_params = [
    "param_preprocessor__text__tfidf__max_features",
    "param_preprocessor__text__tfidf__ngram_range",
    "param_preprocessor__text__tfidf__min_df",
    "param_preprocessor__text__svd__n_components"
]

print("\nRepresentation Parameter Impact:")

representation_impact_summary = []

for param in representation_params:
    impact_table = (
        tuning_results
        .groupby(param)["mean_test_score"]
        .agg(["mean", "max", "count"])
        .sort_values(by="max", ascending=False)
        .round(4)
    )

    print(f"\nPerformance by {param}:")
    display(impact_table)

    best_value = impact_table.index[0]
    best_max_score = impact_table.iloc[0]["max"]

    representation_impact_summary.append({
        "Representation Parameter": param,
        "Best Value Based on Max Score": best_value,
        "Best Max Weighted F1": best_max_score
    })

representation_impact_df = pd.DataFrame(representation_impact_summary)

print("\nSummary of Representation Parameters That Mattered Most:")
display(representation_impact_df)

print("\nFinal Best Representation Settings:")
print("TF-IDF max features:", random_search.best_params_["preprocessor__text__tfidf__max_features"])
print("TF-IDF ngram range:", random_search.best_params_["preprocessor__text__tfidf__ngram_range"])
print("TF-IDF min_df:", random_search.best_params_["preprocessor__text__tfidf__min_df"])
print("SVD components:", random_search.best_params_["preprocessor__text__svd__n_components"])

print("\nFinal Best Boosted Model Settings:")
print("learning_rate:", random_search.best_params_["model__learning_rate"])
print("max_iter:", random_search.best_params_["model__max_iter"])
print("max_leaf_nodes:", random_search.best_params_["model__max_leaf_nodes"])
print("max_depth:", random_search.best_params_["model__max_depth"])
print("l2_regularization:", random_search.best_params_["model__l2_regularization"])

Starting RandomizedSearchCV for joint representation and model tuning...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Best Cross-Validated Weighted F1:
0.6055

Best Parameters:
preprocessor__text__tfidf__ngram_range : (1, 1)
preprocessor__text__tfidf__min_df : 5
preprocessor__text__tfidf__max_features : 1000
preprocessor__text__svd__n_components : 200
model__max_leaf_nodes : 15
model__max_iter : 200
model__max_depth : 5
model__learning_rate : 0.1
model__l2_regularization : 0.01


,Parameter,Best Value
0,preprocessor__text__tfidf__ngram_range,"(1, 1)"
1,preprocessor__text__tfidf__min_df,5
2,preprocessor__text__tfidf__max_features,1000
3,preprocessor__text__svd__n_components,200
4,model__max_leaf_nodes,15
5,model__max_iter,200
6,model__max_depth,5
7,model__learning_rate,0.1
8,model__l2_regularization,0.01



Top 10 Tuning Combinations:


,mean_test_score,std_test_score,mean_train_score,std_train_score,param_preprocessor__text__tfidf__max_features,param_preprocessor__text__tfidf__ngram_range,param_preprocessor__text__tfidf__min_df,param_preprocessor__text__svd__n_components,param_model__learning_rate,param_model__max_iter,param_model__max_leaf_nodes,param_model__max_depth,param_model__l2_regularization
3,0.6055,0.0246,0.9592,0.0046,1000,"(1, 1)",5,200,0.10,200,15,5,0.01
10,0.6037,0.0285,0.9020,0.0202,2000,"(1, 1)",5,200,0.05,300,63,3,0.10
14,0.6035,0.0213,0.8720,0.0488,1000,"(1, 1)",5,50,0.05,200,15,3,0.00
17,0.6014,0.0231,0.9594,0.0045,5000,"(1, 1)",5,50,0.10,300,31,7,0.00
18,0.5982,0.0181,0.9173,0.0282,5000,"(1, 1)",2,200,0.10,100,15,3,0.10
0,0.5978,0.0164,0.8455,0.0323,1000,"(1, 1)",2,50,0.05,300,15,3,0.10
19,0.5916,0.0203,0.9592,0.0047,5000,"(1, 1)",10,200,0.10,300,15,7,0.00
11,0.5910,0.0123,0.9314,0.0046,1000,"(1, 1)",10,200,0.01,200,15,7,0.01
4,0.5843,0.0321,0.8442,0.0355,5000,"(1, 2)",10,50,0.10,200,31,3,0.01
7,0.5821,0.0185,0.8617,0.0260,5000,"(1, 2)",10,100,0.10,100,15,3,0.00



Representation Parameter Impact:

Performance by param_preprocessor__text__tfidf__max_features:


...


Performance by param_preprocessor__text__svd__n_components:


,mean,max,count
param_preprocessor__text__svd__n_components,,,
200,0.5916,0.6055,6
50,0.5873,0.6035,7
100,0.5726,0.5821,7



Summary of Representation Parameters That Mattered Most:


,Representation Parameter,Best Value Based on Max Score,Best Max Weighted F1
0,param_preprocessor__text__tfidf__max_features,1000,0.6055
1,param_preprocessor__text__tfidf__ngram_range,"(1, 1)",0.6055
2,param_preprocessor__text__tfidf__min_df,5,0.6055
3,param_preprocessor__text__svd__n_components,200,0.6055



Final Best Representation Settings:
TF-IDF max features: 1000
TF-IDF ngram range: (1, 1)
TF-IDF min_df: 5
SVD components: 200

Final Best Boosted Model Settings:
learning_rate: 0.1
max_iter: 200
max_leaf_nodes: 15
max_depth: 5
l2_regularization: 0.01


#### Hyperparameter Tuning Results and Interpretation


For hyperparameter tuning, we used RandomizedSearchCV to jointly tune both the text representation and the boosted model. This satisfies the graduate-level tuning requirement because the search included TF-IDF parameters, SVD parameters, and model hyperparameters in one pipeline.

The best cross-validated weighted F1 score was 0.6055. This is slightly higher than the  Voting Classifier result of 0.60, but the difference is only 0.0055, which is smaller than the cross-validation variability of approximately 0.02. Therefore, this result should not be interpreted as a clear performance improvement. Instead, the tuned HistGradientBoosting model should be described as competitive with the best earlier model.

The best text representation used 1000 TF-IDF max features, unigram-only text features with n-gram range (1, 1), minimum document frequency of 5, and 200 SVD components. The unigram-only setting suggests that single-word signals were the most stable representation for this dataset. Although bigrams and trigrams can capture useful phrase-level meaning, they did not improve performance here. This is likely because the dataset has only about 2,000 rows and a relatively small vocabulary. Adding bigrams and trigrams creates many sparse features that appear too rarely to generalize well. Since the pipeline also compresses the text representation using TruncatedSVD, rare phrase-level signals may not be preserved effectively.

The best boosted model settings were learning_rate = 0.1, max_iter = 200, max_leaf_nodes = 15, max_depth = 5, and l2_regularization = 0.01. These settings indicate that a relatively shallow, regularized boosted model performed best. This is useful because it helps reduce overfitting while still capturing nonlinear patterns in the combined text and structured features.

The representation impact tables show that the most important representation choices were TF-IDF n-gram range and SVD components. The unigram-only setting had the strongest average and maximum performance, while 200 SVD components reached the highest maximum weighted F1. Overall, the tuning results show that text representation choices had a meaningful effect on model performance, not just the classifier hyperparameters. The final tuned model should be interpreted as competitive rather than clearly better than the earlier best model.